In [28]:
import math
import random
import pandas as pd

In [29]:
def sigmoid(x):
    x = max(min(x, 500), -500)
    return 1 / (1 + math.exp(-x))

def sigmoid_derivative(y):
    return y * (1 - y)

In [30]:
def initialize_network():
    """Initializes the MLP with a 3-5-1 topology."""
    # w1: Weights from 3 input neurons to 5 hidden neurons (5 lists of 3 weights)
    w1 = [[random.uniform(-0.5, 0.5) for _ in range(3)] for _ in range(5)]
    
    # w2: Weights from 5 hidden neurons to 1 output neuron (1 list of 5 weights)
    w2 = [random.uniform(-0.5, 0.5) for _ in range(5)]
    
    # Biases for the hidden and output layers
    b_hidden = [random.uniform(-0.5, 0.5) for _ in range(5)]
    b_output = random.uniform(-0.5, 0.5)
    
    network = {
        'w1': w1,
        'w2': w2,
        'b_hidden': b_hidden,
        'b_output': b_output
    }
    return network

In [31]:
def forward_pass(network, inputs):
    w1, w2 = network['w1'], network['w2']
    b_hidden, b_output = network['b_hidden'], network['b_output']

    hidden_activations = []
    for i in range(5):
        z_hidden = sum(inputs[j] * w1[i][j] for j in range(len(inputs))) + b_hidden[i]
        hidden_activations.append(sigmoid(z_hidden))

    z_output = sum(hidden_activations[j] * w2[j] for j in range(5)) + b_output
    output_activation = sigmoid(z_output)
    return hidden_activations, output_activation

In [32]:

def backward_pass(network, hidden_activations, output_activation, target):
    w2 = network['w2']
    output_error = target - output_activation
    output_delta = output_error * sigmoid_derivative(output_activation)

    hidden_deltas = []
    for j in range(5):
        error = output_delta * w2[j]
        delta = error * sigmoid_derivative(hidden_activations[j])
        hidden_deltas.append(delta)

    return hidden_deltas, output_delta

In [33]:
def update_weights(network, inputs, hidden_activations, hidden_deltas, output_delta, learning_rate):

    for j in range(len(network['w2'])):
        network['w2'][j] += learning_rate * output_delta * hidden_activations[j]
    network['b_output'] += learning_rate * output_delta
    for i in range(len(network['w1'])):
        for j in range(len(inputs)):
            network['w1'][i][j] += learning_rate * hidden_deltas[i] * inputs[j]
        network['b_hidden'][i] += learning_rate * hidden_deltas[i]

In [34]:
def train_network(network, training_data, epochs, learning_rate):
    print("--- Starting Network Training ---")
    for epoch in range(epochs):
        sum_error = 0
        for inputs, target in training_data:
            # Step 1: Forward Pass
            hidden_activations, output_activation = forward_pass(network, inputs)
            
            # Add to error for monitoring
            sum_error += (target[0] - output_activation)**2
            
            # Step 2: Backward Pass
            hidden_deltas, output_delta = backward_pass(network, hidden_activations, output_activation, target[0])
            
            # Step 3: Update Weights
            update_weights(network, inputs, hidden_activations, hidden_deltas, output_delta, learning_rate)

        if epoch % 1000 == 0 or epoch == epochs - 1:
            print(f'> Epoch={epoch}, Learning Rate={learning_rate:.2f}, Error={sum_error:.4f}')
    print("--- Training Complete ---")

In [35]:
BMI = pd.read_csv('../bmi.csv',header=0,names=['Gender','Height','Weight','Index'])
BMI

,Gender,Height,Weight,Index
0,Male,174,96,4
1,Male,189,87,2
2,Female,185,110,4
3,Female,195,104,3
4,Male,149,61,3
...,...,...,...,...
495,Female,150,153,5
496,Female,184,121,4
497,Female,141,136,5
498,Male,150,95,5


In [36]:
df= pd.DataFrame()
df['Gender'] = BMI['Gender'].apply(lambda x: 0.1 if str(x).lower() == 'male' else 0.2)
df['Height'] = BMI['Height']/1000
df['Weight'] = BMI['Weight']/100
df['Index'] = BMI['Index']/10
df.head()

,Gender,Height,Weight,Index
0,0.1,0.174,0.96,0.4
1,0.1,0.189,0.87,0.2
2,0.2,0.185,1.10,0.4
3,0.2,0.195,1.04,0.3
4,0.1,0.149,0.61,0.3


In [37]:
if __name__ == "__main__":
    norm_params = {
        'Height': {'mean': df['Height'].mean(), 'std': df['Height'].std()},
        'Weight': {'mean': df['Weight'].mean(), 'std': df['Weight'].std()},
    }       
    scale_params = {
        'Index': {'min': df['Index'].min(), 'max': df['Index'].max()}
    }

    training_dataset = []
    for _, row in df.iterrows():
        inputs = [
            row['Gender'],
            (row['Height'] - norm_params['Height']['mean']) / norm_params['Height']['std'],
            (row['Weight'] - norm_params['Weight']['mean']) / norm_params['Weight']['std']
        ]
        index_val = row['Index']
        scaled_index = 0.1 + 0.8 * (index_val - scale_params['Index']['min']) / (scale_params['Index']['max'] - scale_params['Index']['min'])
        target = [scaled_index]
        training_dataset.append((inputs, target))

    network = initialize_network()
    learning_rate = 0.1
    epochs = 10000
    train_network(network, training_dataset, epochs+1, learning_rate)


--- Starting Network Training ---
> Epoch=0, Learning Rate=0.10, Error=20.5074
> Epoch=1000, Learning Rate=0.10, Error=1.3983
> Epoch=2000, Learning Rate=0.10, Error=1.3444
> Epoch=3000, Learning Rate=0.10, Error=1.2986
> Epoch=4000, Learning Rate=0.10, Error=1.2376
> Epoch=5000, Learning Rate=0.10, Error=1.1381
> Epoch=6000, Learning Rate=0.10, Error=1.0498
> Epoch=7000, Learning Rate=0.10, Error=1.0121
> Epoch=8000, Learning Rate=0.10, Error=0.9960
> Epoch=9000, Learning Rate=0.10, Error=0.9863
> Epoch=10000, Learning Rate=0.10, Error=0.9794
--- Training Complete ---


In [38]:
print("\n--- Making a Prediction ---")
input_gender= 1
input_height_cm = 189
input_weight_kg = 87
test_gender = input_gender/10
test_height_cm = input_height_cm/1000
test_weight_kg = input_weight_kg/100

norm_height = (test_height_cm - norm_params['Height']['mean']) / norm_params['Height']['std']
norm_weight = (test_weight_kg - norm_params['Weight']['mean']) / norm_params['Weight']['std']
test_input = [test_gender, norm_height, norm_weight]
_, scaled_prediction = forward_pass(network, test_input)

# Reverse scaling to original target space (which is Index/10)
unscaled_pred = (scaled_prediction - 0.1) / 0.8
predicted_index_over_10 = (unscaled_pred * (scale_params['Index']['max'] - scale_params['Index']['min']) + scale_params['Index']['min'])

# Convert to Index class (approx.)
predicted_index_class = (predicted_index_over_10 * 10)

print(f"Input: Gender = Male, Height = {input_height_cm} cm, Weight = {input_weight_kg} kg")
print(f"Predicted BMI Index (value/10): {predicted_index_over_10:.2f}")
print(f"Predicted BMI Index: {predicted_index_class:.1f}")


--- Making a Prediction ---
Input: Gender = Male, Height = 189 cm, Weight = 87 kg
Predicted BMI Index (value/10): 0.23
Predicted BMI Index: 2.3
